# Universal ComfyUI Colab — managers + core nodes only

This notebook installs **ComfyUI** with both managers and a core node pack,
then launches it through a **Cloudflare Tunnel**. It downloads **no models** —
after launch, browse to the URL and install exactly the models you need:

| Tool | What it does |
|---|---|
| **ComfyUI-Manager** (⚙ Manager button in the UI) | Install/update custom nodes |
| **ComfyUI-Model-Manager** (`/model-manager` page or sidebar) | Browse & download models (HF/Civitai), tokens pre-wired from your Secrets |

Quick start:
1. Add `HF_TOKEN` (required for gated models) and optionally `CIVITAI_API_TOKEN`
   to the 🔑 Secrets panel with notebook access ON.
2. Run all cells top to bottom.
3. Open the tunnel URL, use the managers to fetch models, generate.

Core node pack: ComfyUI-GGUF, KJNodes, rgthree-comfy,
Workflow-Models-Downloader.

In [ ]:
# @title 1) Tokens (Colab Secrets / environment only)
# Tokens are read from Colab Secrets (panel 🔑) or process environment.
# No interactive prompts: if HF_TOKEN is missing the cell fails fast with
# clear instructions instead of hanging or raising NameError later.
import os

!pip install -q -U huggingface_hub
from huggingface_hub import login

try:
    from google.colab import userdata
except Exception:
    userdata = None


def colab_secret(name: str) -> str:
    """Read a secret from env first, then Colab Secrets."""
    value = os.environ.get(name, "").strip()
    if not value and userdata is not None:
        try:
            raw = userdata.get(name) or ""
        except Exception:
            raw = ""
        if isinstance(raw, dict):
            raw = raw.get("value") or raw.get("token") or next(iter(raw.values()), "")
        value = str(raw).strip()
    return value


HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found.\n"
        "  • Colab: open the 🔑 Secrets panel, add HF_TOKEN (notebook access ON), rerun.\n"
        "  • Jupyter locally: export HF_TOKEN=... before starting."
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF auth: OK")

CIVITAI_API_TOKEN = colab_secret("CIVITAI_API_TOKEN")
if CIVITAI_API_TOKEN:
    os.environ["CIVITAI_API_TOKEN"] = CIVITAI_API_TOKEN
    print("Civitai auth: OK")
else:
    print("Civitai token not set (optional); Civitai downloads may return 403.")

print("Done.")


In [ ]:
# @title 2) Install ComfyUI + Managers + node pack (incl. rgthree)
# Installs ComfyUI, ComfyUI-Manager, ComfyUI-Model-Manager, ComfyUI-GGUF,
# KJNodes and rgthree-comfy. Idempotent: safe to rerun.
import os
import subprocess
import sys


def sh(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)


def pip_ok(spec):
    rc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", spec], check=False
    ).returncode
    return rc == 0


# Swap protects small-RAM Colab VMs during dependency resolution.
if not os.path.exists("/swapfile"):
    print("Creating swap (8GB)...")
    !sudo fallocate -l 8G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    !sudo swapon /swapfile
    print("Swap enabled.")

!apt-get -y update -qq
!apt-get -y install -qq aria2 ffmpeg cloudflared

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

COMFY_ROOT = "/content/ComfyUI"

# ── ComfyUI core ──────────────────────────────────────────────────────────────
if not os.path.exists(COMFY_ROOT):
    print("Cloning ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY_ROOT}

print("Installing ComfyUI requirements...")
!pip install -q -U pip
!pip install -q -r {COMFY_ROOT}/requirements.txt

# ── Optional attention accelerators (GPU-aware, never fatal) ────────────────
ENABLE_ACCEL_PACK = True
if ENABLE_ACCEL_PACK:
    try:
        import torch

        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        cc_major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
        print(f"GPU detected: {gpu_name} (sm{cc_major}0)")
        print("xformers:", "OK" if pip_ok("xformers>=0.0.28.post3") else "FAILED (continuing)")
        if cc_major >= 8:
            print("flash-attn:", "OK" if pip_ok("flash-attn>=2.7.0.post2") else "FAILED (continuing)")
            print("sageattention:", "OK" if pip_ok("sageattention>=2.1.1") else "FAILED (continuing)")
        else:
            print("Skipping flash-attn/sageattention on sm<8 (T4-safe xformers path).")
    except Exception as exc:
        print("Accelerator pack skipped:", exc)

# ── Custom nodes ──────────────────────────────────────────────────────────────
NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    # Separate model manager (different project from ComfyUI-Manager):
    "ComfyUI-Model-Manager": "https://github.com/hayden-cn/ComfyUI-Model-Manager.git",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "comfyui-kjnodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Workflow-Models-Downloader": "https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git",
}
for folder, repo in NODES.items():
    target = f"{COMFY_ROOT}/custom_nodes/{folder}"
    if not os.path.exists(target):
        print("Cloning", folder, "...")
        !git clone --depth 1 {repo} {target}
    req = f"{target}/requirements.txt"
    if os.path.exists(req):
        !pip install -q -r {req}

print("\nInstalled node packs:")
for folder in sorted(os.listdir(f"{COMFY_ROOT}/custom_nodes")):
    if not folder.startswith("__"):
        print("  -", folder)
print("\nComfyUI + Managers + node pack ready.")


In [ ]:
# @title 3) Bridge tokens into Model Manager
# ComfyUI-Model-Manager keeps download credentials in its private.key pickle.
# This cell writes the Colab/env tokens there so downloads made through the
# Model Manager UI (Hugging Face / Civitai) are authenticated automatically.
import os
import pickle
from pathlib import Path

COMFY_ROOT = Path("/content/ComfyUI")
MODEL_MANAGER_DIR = COMFY_ROOT / "custom_nodes" / "ComfyUI-Model-Manager"
assert MODEL_MANAGER_DIR.exists(), "Model Manager is not installed — run the install cell first."

manager_key_file = MODEL_MANAGER_DIR / "private.key"
manager_keys = {}
if manager_key_file.exists():
    try:
        with manager_key_file.open("rb") as stream:
            loaded = pickle.load(stream)
        if isinstance(loaded, dict):
            manager_keys.update(loaded)
    except Exception:
        print("Existing key file unreadable; recreating it.")

hf_token = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
civitai_token = os.environ.get("CIVITAI_API_TOKEN", "").strip()
if hf_token:
    manager_keys["huggingface"] = hf_token
if civitai_token:
    manager_keys["civitai"] = civitai_token

if manager_keys:
    tmp = manager_key_file.with_suffix(".private.key.tmp")
    with tmp.open("wb") as stream:
        pickle.dump(manager_keys, stream, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(manager_key_file)
print(
    "Token bridge:",
    "HF=" + ("OK" if hf_token else "not set"),
    "| Civitai=" + ("OK" if civitai_token else "not set"),
)

## 5) Launch
Starts ComfyUI and prints the public Cloudflare Tunnel URL.
Then open **Model Manager** in the ComfyUI UI to download models.


In [ ]:
# @title 5) Launch ComfyUI + self-healing Cloudflare Tunnel
# Boots ComfyUI, opens a Cloudflare Quick Tunnel (no account, no password page),
# health-checks the public URL (/system_stats, Save Image /view, frontend assets)
# and automatically recreates the tunnel if it dies.
import base64, os, queue, re, shutil, socket, subprocess, threading, time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests

LOW_VRAM_STABLE = False  # True: slower but safer for large images on free Colab
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
OUTPUT_DIR = COMFY_ROOT / "output"
COMFY_PORT = 8188

# ── Model Manager token bridge (keys live in its private.key pickle) ────────
MODEL_MANAGER_DIR = COMFY_ROOT / "custom_nodes" / "ComfyUI-Model-Manager"
if MODEL_MANAGER_DIR.exists():
    import pickle

    manager_key_file = MODEL_MANAGER_DIR / "private.key"
    manager_keys = {}
    if manager_key_file.exists():
        try:
            with manager_key_file.open("rb") as stream:
                loaded = pickle.load(stream)
            if isinstance(loaded, dict):
                manager_keys.update(loaded)
        except Exception:
            print("Existing Model Manager key file was unreadable; recreating it.")

    hf_for_manager = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
    civitai_for_manager = os.environ.get("CIVITAI_API_TOKEN", "").strip()
    if hf_for_manager:
        manager_keys["huggingface"] = hf_for_manager
    if civitai_for_manager:
        manager_keys["civitai"] = civitai_for_manager
    if manager_keys:
        manager_key_tmp = manager_key_file.with_suffix(".private.key.tmp")
        with manager_key_tmp.open("wb") as stream:
            pickle.dump(manager_keys, stream, protocol=pickle.HIGHEST_PROTOCOL)
        manager_key_tmp.replace(manager_key_file)
    print(
        "Model Manager token bridge:",
        "HF=" + ("yes" if hf_for_manager else "existing/none"),
        "Civitai=" + ("yes" if civitai_for_manager else "existing/none"),
    )
else:
    print("⚠ ComfyUI-Model-Manager not found — run the install cell first.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes started by any earlier launch-cell version.
old_stop = globals().get("_TUNNEL_STOP")
if old_stop is not None:
    old_stop.set()
for process_name in ("_TUNNEL_PROC", "_COMFY_PROC"):
    stop_process(globals().get(process_name))
old_log = globals().get("_COMFY_LOG")
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def ensure_cloudflared():
    """cloudflared must be on PATH (installed by the setup cell; apt fallback)."""
    if shutil.which("cloudflared") is not None:
        return
    print("Installing cloudflared...")
    subprocess.run(["apt-get", "y", "-qq", "install", "cloudflared"], check=False)
    if shutil.which("cloudflared") is None:
        # Direct binary download as last resort.
        arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(os.uname().machine, "amd64")
        url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
        subprocess.run(["curl", "-fsSL", "-o", "/usr/local/bin/cloudflared", url], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    if shutil.which("cloudflared") is None:
        raise RuntimeError("cloudflared installation failed.")


ensure_cloudflared()

comfy_args = [
    "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
    "--enable-cors-header", "*", "--output-directory", str(OUTPUT_DIR),
]
comfy_args += (
    ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
    if LOW_VRAM_STABLE else ["--lowvram", "--preview-method", "auto"]
)
_COMFY_LOG = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def local_comfy_ready(timeout=300):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log")
        try:
            with socket.create_connection(("127.0.0.1", COMFY_PORT), timeout=2):
                response = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
                if response.ok:
                    return True
        except (OSError, requests.RequestException):
            time.sleep(2)
    return False


if not local_comfy_ready():
    raise TimeoutError("ComfyUI did not become ready within 300 seconds.")
print("ComfyUI is ready locally. Output:", OUTPUT_DIR)


def read_process_lines(proc, lines):
    for line in iter(proc.stdout.readline, ""):
        lines.put(line.rstrip())


def start_cloudflare_tunnel(timeout=120):
    """Start a Quick Tunnel and return (proc, public_url, recent_output)."""
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}",
         "--no-autoupdate", "--protocol", "http2"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = queue.Queue()
    threading.Thread(target=read_process_lines, args=(proc, lines), daemon=True).start()
    recent = []
    deadline = time.time() + timeout
    while time.time() < deadline and proc.poll() is None:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            continue
        recent = (recent + [line])[-12:]
        match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line, re.I)
        if match:
            return proc, match.group(0), recent
    stop_process(proc)
    return None, None, recent


def public_comfy_ready(url, timeout=15):
    try:
        response = requests.get(url.rstrip("/") + "/system_stats", timeout=timeout)
        if not response.ok:
            return False
        payload = response.json()
        return isinstance(payload, dict) and ("system" in payload or "devices" in payload)
    except (requests.RequestException, ValueError):
        return False


def local_backend_responding(timeout=5):
    try:
        return requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=timeout).ok
    except requests.RequestException:
        return False


def public_tunnel_responding(url, timeout=15):
    try:
        response = requests.get(url.rstrip("/") + "/", timeout=timeout)
        return response.ok and ("<html" in response.text[:4096].lower())
    except requests.RequestException:
        return False


def public_image_route_ready(url):
    # Verify the exact /view route used by Save Image previews.
    probe_name = "_cf_tunnel_image_probe.png"
    probe_path = OUTPUT_DIR / probe_name
    probe_png = base64.b64decode(
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII="
    )
    try:
        probe_path.write_bytes(probe_png)
        response = requests.get(
            url.rstrip("/") + "/view",
            params={"filename": probe_name, "type": "output", "subfolder": ""},
            timeout=15,
        )
        return response.ok and response.content.startswith(b"\x89PNG")
    except requests.RequestException:
        return False
    finally:
        probe_path.unlink(missing_ok=True)


_TUNNEL_STOP = threading.Event()
_TUNNEL_PROC = None
_LAST_URL = None


def tunnel_supervisor():
    global _TUNNEL_PROC, _LAST_URL
    while not _TUNNEL_STOP.is_set():
        print("Starting Cloudflare Tunnel...")
        proc, url, recent = start_cloudflare_tunnel()
        if proc is None:
            print("Cloudflare Tunnel failed:", recent[-4:] or ["no output"])
            _TUNNEL_STOP.wait(10)
            continue

        _TUNNEL_PROC = proc
        _LAST_URL = url
        print("\n" + "=" * 70)
        print("ComfyUI public URL:", url)
        print("=" * 70)

        # Startup diagnostics (non-gating: warmup may lag a few seconds).
        if public_comfy_ready(url):
            print("/system_stats via tunnel: OK")
        else:
            print("/system_stats via tunnel: warming up; refresh once if needed.")
        if public_image_route_ready(url):
            print("Save Image /view route: OK")
        else:
            print("Save Image /view route: will re-check next cycle.")

        public_failure_since = None
        local_busy_reported = False
        while not _TUNNEL_STOP.wait(5):
            if _COMFY_PROC.poll() is not None:
                print("ComfyUI stopped. Inspect /content/comfyui.log")
                stop_process(proc)
                return
            if proc.poll() is not None:
                print("Cloudflare Tunnel process exited; restarting immediately...")
                break
            if public_tunnel_responding(url):
                public_failure_since = None
                local_busy_reported = False
                continue
            if not local_backend_responding():
                # Sampling can stall ComfyUI HTTP responses; keep the URL.
                public_failure_since = None
                if not local_busy_reported:
                    print("ComfyUI busy locally; keeping the current tunnel URL.")
                    local_busy_reported = True
                continue
            local_busy_reported = False
            if public_failure_since is None:
                public_failure_since = time.time()
                print("Tunnel temporarily unhealthy; waiting before recreation...")
            elif time.time() - public_failure_since >= 30:
                print("Tunnel stayed unhealthy for 30 seconds; recreating it...")
                stop_process(proc)
                break
        stop_process(proc)


print("Cloudflare Tunnel supervisor is running. Stop this cell to close ComfyUI.")
try:
    tunnel_supervisor()
except KeyboardInterrupt:
    print("Stopping Cloudflare Tunnel and ComfyUI...")
finally:
    _TUNNEL_STOP.set()
    stop_process(_TUNNEL_PROC)
    stop_process(_COMFY_PROC)
    try:
        _COMFY_LOG.close()
    except Exception:
        pass
